<a href="https://colab.research.google.com/github/Nethika-Alagarathnam/AnimatedSite/blob/main/TF_IDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

human_patterns = [
    'AI can\'t replace human and I hope so', 'When we are writing research papers we have to consider about plagiarism.',
    'Instead of going outside I would prefer staying at home these days.', 'After finish writing the full paper we have to start writing abstract.',
    'Traditional selection processes for internships often face challenges.', 'MySQL is query based database system.',
    'I made tea and then accidentally spilled it on my notebook.', 'I stayed home because the weather was bad',
    'I tried studying but felt very distracted', 'I was nervous before presenting my work'
]

def generate_human_texts(n=1000):
    texts = []
    while len(texts) < n:
        base = random.choice(human_patterns)
        if 5 <= len(base.split()) <= 15: texts.append(base)
    return texts

def generate_ai_texts(n=1000):
    ai_samples = [
        'AI models can generate human-like text by predicting the next word in sequence.',
        'The effectiveness of machine learning depends on the quality of training data used.',
        'Deep learning architectures like transformers have significantly improved text generation capabilities.',
        'Logistic regression is a lightweight algorithm often used for binary classification tasks.'
    ]
    texts = []
    while len(texts) < n:
        text = random.choice(ai_samples)
        if 5 <= len(text.split()) <= 15: texts.append(text)
    return texts

data = pd.DataFrame({'text': generate_ai_texts(1000) + generate_human_texts(1000), 'label': ['AI']*1000 + ['Human']*1000})
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = vectorizer.fit_transform(data['text'])
y = data['label']

param_grid = {'C': [0.1, 1, 10, 100], 'solver': ['liblinear']}
grid_search = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5)
grid_search.fit(X_tfidf, y)

cv_scores = cross_val_score(grid_search.best_estimator_, X_tfidf, y, cv=5)

X_train, X_test, y_train, y_test = train_test_split(data['text'], y, test_size=0.2, random_state=42)
model = grid_search.best_estimator_
model.fit(vectorizer.transform(X_train), y_train)
y_pred = model.predict(vectorizer.transform(X_test))

misclassified = np.where(y_test != y_pred)[0]

print(f'Best Params: {grid_search.best_params_}')
print(f'CV Mean: {cv_scores.mean():.4f}')
print(f'CV Std: {cv_scores.std():.4f}')
print(f'Misclassified Count: {len(misclassified)}')


Best Params: {'C': 0.1, 'solver': 'liblinear'}
CV Mean: 1.0000
CV Std: 0.0000
Misclassified Count: 0
